# Lab 22-09 · Cell Models, CC/CV Charging and Gauging Errors

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import battery as bt

Q = 1080.0   # teaching cell, coulombs


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The pulse and relaxation

*Predict* the voltage at the first instant, after 20 s, and 40 s after a 40 s, 0.5 A pulse ends.

In [2]:
z0 = 0.5 / 0.9
t, z, v1, v = bt.simulate_cell(lambda t: 0.5 if 1.0 <= t < 41.0 else 0.0, 81.0, Q, z0, n=1621)
plt.figure(); plt.plot(t, bt.ocv_linear(z), "--", label="OCV"); plt.plot(t, v, label="terminal")
plt.xlabel("time (s)"); plt.ylabel("V"); plt.legend(); plt.show()
print(f"after 40 s of rest: {v[-1]:.4f} V, OCV {bt.ocv_linear(z[-1]):.4f} V, "
      f"voltage-gauge SOC error {100*(bt.ocv_linear(z[-1]) - v[-1])/0.9:.2f} %")
s = ngspice("pulse.cir")
print({kk: s[kk] for kk in ("va", "vd", "vrest")})

after 40 s of rest: 3.7775 V, OCV 3.7833 V, voltage-gauge SOC error 0.65 %
{'va': 3.699996, 'vd': 3.660061, 'vrest': 3.777481}


## Part 2 · CC/CV

*Predict* the CC time, total time and final SOC at 1C, and the total time at 2C.

In [3]:
for i_cc in (0.3, 0.6):
    z_cv, t_cc, tau, t_cv, z_t = bt.cccv_linear(Q, i_cc, 0.03, 0.30)
    print(f"I_cc {i_cc} A: CC ends at z = {z_cv:.3f} after {t_cc/60:.1f} min; tau {tau:.0f} s; "
          f"CV {t_cv/60:.1f} min; total {(t_cc + t_cv)/60:.1f} min; z_term {z_t:.3f}")
s = ngspice("cccv.cir")
print({kk: s[kk] for kk in ("tcc", "tterm", "zterm", "tterm2")})

I_cc 0.3 A: CC ends at z = 0.900 after 54.0 min; tau 360 s; CV 13.8 min; total 67.8 min; z_term 0.990
I_cc 0.6 A: CC ends at z = 0.800 after 24.0 min; tau 360 s; CV 18.0 min; total 42.0 min; z_term 0.990
{'tcc': 3240.105, 'tterm': 4068.931, 'zterm': 0.99, 'tterm2': 2518.463}


## Part 3 · Which simplifications matter?

*Predict* the effect of (i) the RC branch and (ii) an S-shaped OCV on the 1C charge time, and of the S-shaped OCV on Example 22-09.3's cold penalty.

In [4]:
t_cc, t_end, z_end = bt.simulate_cccv_rc(Q, 0.3, 0.03)
print(f"RC branch included: charge {t_end/60:.1f} min (quasi-static: 67.8 min)")
t_cc, t_end, z_end = bt.simulate_cccv(Q, 0.3, 0.03, 0.30, ocv=bt.ocv_curved)
print(f"S-shaped OCV: charge {t_end/60:.1f} min, z_term {z_end:.3f}")
for r, label in ((0.30, "25 C"), (1.20, "-10 C")):
    z_cut = optimize.brentq(lambda zz: float(bt.ocv_curved(zz)) - 3.0 - 0.3 * r, 0, 1)
    print(f"S-shaped OCV, {label}: usable {100*(1 - z_cut):.1f} %   (straight line: {100*bt.usable_fraction_linear(0.3, r):.0f} %)")
zz = np.linspace(0, 1, 200)
plt.figure(); plt.plot(zz, bt.ocv_linear(zz), label="straight line"); plt.plot(zz, bt.ocv_curved(zz), label="S-shaped")
plt.xlabel("SOC"); plt.ylabel("OCV (V)"); plt.legend(); plt.show()

RC branch included: charge 68.0 min (quasi-static: 67.8 min)
S-shaped OCV: charge 63.5 min, z_term 0.996
S-shaped OCV, 25 C: usable 97.9 %   (straight line: 90 %)
S-shaped OCV, -10 C: usable 86.6 %   (straight line: 60 %)


## Part 4 · Gauging errors

*Predict* how far a coulomb counter with a 10 µA offset drifts in a week, and how a voltage gauge's estimate moves with each load pulse.

In [5]:
days = np.linspace(0, 7, 200)
drift = bt.coulomb_offset_error(10e-6, days * 86400) / 3.6   # mAh
plt.figure(); plt.plot(days, drift); plt.xlabel("days"); plt.ylabel("counter error (mAh)"); plt.show()
print(f"after 7 days: {drift[-1]:.2f} mAh = {100*drift[-1]/300:.2f} % of 300 mAh")
for i in (0.0, 0.1, 0.3, 0.6):
    print(f"load {i:.1f} A: voltage-gauge SOC error {100*bt.voltage_gauge_error(i, 0.30, 0.90):.0f} points")

after 7 days: 1.68 mAh = 0.56 % of 300 mAh
load 0.0 A: voltage-gauge SOC error 0 points
load 0.1 A: voltage-gauge SOC error 3 points
load 0.3 A: voltage-gauge SOC error 10 points
load 0.6 A: voltage-gauge SOC error 20 points


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.